In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline

# Load dataset
df_train = pd.read_csv('../resources/train_preprocessing_awal_shallow.csv')
df_val = pd.read_csv('../resources/val_preprocessing_awal_shallow.csv')
df_test = pd.read_csv('../resources/test_preprocessing_awal_shallow.csv')

X_train = df_train['Pesan_Clean'].fillna('')
y_train = df_train['Label Toxic']

X_val = df_val['Pesan_Clean'].fillna('')
y_val = df_val['Label Toxic']

X_test = df_test['Pesan_Clean'].fillna('')
y_test = df_test['Label Toxic']

In [2]:
# LinearSVM dengan GridSearchCV
X_combined = pd.concat([X_train, X_val], axis=0)
y_combined = pd.concat([y_train, y_val], axis=0)

split_index = [-1] * len(X_train) + [0] * len(X_val)
pds = PredefinedSplit(test_fold=split_index)

# Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svm', LinearSVC(class_weight='balanced', random_state=42, max_iter=2000))
])

# Param
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [5000, 10000],
    'svm__C': [0.1, 1, 10]
}

print("Memulai GridSearchCV... (Ini akan mencoba banyak kombinasi)")
grid_search = GridSearchCV(pipeline, param_grid, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_search.fit(X_combined, y_combined) 

print(f"\nParameter Terbaik SVM: {grid_search.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=grid_search.best_params_['tfidf__ngram_range'],
        max_features=grid_search.best_params_['tfidf__max_features']
    )),
    ('svm', LinearSVC(
        C=grid_search.best_params_['svm__C'],
        class_weight='balanced', 
        random_state=42, 
        max_iter=2000
    ))
])

best_svm_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val = best_svm_model.predict(X_val)
print(classification_report(y_val, y_pred_val))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test = best_svm_model.predict(X_test)
print(classification_report(y_test, y_pred_test))

Memulai GridSearchCV... (Ini akan mencoba banyak kombinasi)
Fitting 1 folds for each of 12 candidates, totalling 12 fits

Parameter Terbaik SVM: {'svm__C': 0.1, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

      normal       0.97      0.95      0.96      9746
       toxic       0.59      0.67      0.63       996

    accuracy                           0.93     10742
   macro avg       0.78      0.81      0.79     10742
weighted avg       0.93      0.93      0.93     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

      normal       0.99      0.97      0.98      9262
       toxic       0.37      0.60      0.46       280

    accuracy                           0.96      9542
   macro avg       0.68      0.78      0.72      9542
weighted avg       0.97      0.96      0

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import classification_report
import pandas as pd

X_combined = pd.concat([X_train, X_val], axis=0)
y_combined = pd.concat([y_train, y_val], axis=0)

split_index = [-1] * len(X_train) + [0] * len(X_val)
pds = PredefinedSplit(test_fold=split_index)

In [4]:
print("--- Mengoptimasi Logistic Regression ---")
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))
])

param_lr = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'lr__C': [0.1, 1, 10],
    'lr__solver': ['liblinear', 'lbfgs']
}

grid_lr = GridSearchCV(pipe_lr, param_lr, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_lr.fit(X_combined, y_combined)

print(f"\nBest LR Params: {grid_lr.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_lr_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=grid_lr.best_params_['tfidf__ngram_range']
    )),
    ('lr', LogisticRegression(
        C=grid_lr.best_params_['lr__C'],
        solver=grid_lr.best_params_['lr__solver'],
        class_weight='balanced', 
        max_iter=1000
    ))
])

best_lr_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val_lr = best_lr_model.predict(X_val)
print(classification_report(y_val, y_pred_val_lr))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test_lr = best_lr_model.predict(X_test)
print(classification_report(y_test, y_pred_test_lr))

--- Mengoptimasi Logistic Regression ---
Fitting 1 folds for each of 12 candidates, totalling 12 fits

Best LR Params: {'lr__C': 10, 'lr__solver': 'liblinear', 'tfidf__ngram_range': (1, 1)}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

      normal       0.96      0.98      0.97      9746
       toxic       0.74      0.60      0.66       996

    accuracy                           0.94     10742
   macro avg       0.85      0.79      0.81     10742
weighted avg       0.94      0.94      0.94     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

      normal       0.98      0.98      0.98      9262
       toxic       0.48      0.49      0.49       280

    accuracy                           0.97      9542
   macro avg       0.73      0.74      0.74      9542
weighted avg       0.97      0.97      0.97      9542



In [5]:
print("--- Mengoptimasi Naive Bayes ---")
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

param_nb = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'nb__alpha': [0.01, 0.1, 1, 10]
}

grid_nb = GridSearchCV(pipe_nb, param_nb, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_nb.fit(X_combined, y_combined)

print(f"\nBest NB Params: {grid_nb.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_nb_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=grid_nb.best_params_['tfidf__ngram_range']
    )),
    ('nb', MultinomialNB(
        alpha=grid_nb.best_params_['nb__alpha']
    ))
])

best_nb_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val_nb = best_nb_model.predict(X_val)
print(classification_report(y_val, y_pred_val_nb))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test_nb = best_nb_model.predict(X_test)
print(classification_report(y_test, y_pred_test_nb))

--- Mengoptimasi Naive Bayes ---
Fitting 1 folds for each of 8 candidates, totalling 8 fits

Best NB Params: {'nb__alpha': 0.01, 'tfidf__ngram_range': (1, 1)}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

      normal       0.94      1.00      0.97      9746
       toxic       0.91      0.35      0.50       996

    accuracy                           0.94     10742
   macro avg       0.92      0.67      0.73     10742
weighted avg       0.93      0.94      0.92     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

      normal       0.97      1.00      0.99      9262
       toxic       0.67      0.15      0.24       280

    accuracy                           0.97      9542
   macro avg       0.82      0.57      0.62      9542
weighted avg       0.97      0.97      0.96      9542



In [6]:
print("--- Mengoptimasi Random Forest ---")
pipe_rf = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_rf = {
    'tfidf__max_features': [5000], 
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 20]
}

grid_rf = GridSearchCV(pipe_rf, param_rf, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_rf.fit(X_combined, y_combined)

print(f"\nBest RF Params: {grid_rf.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_rf_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=grid_rf.best_params_['tfidf__max_features']
    )),
    ('rf', RandomForestClassifier(
        n_estimators=grid_rf.best_params_['rf__n_estimators'],
        max_depth=grid_rf.best_params_['rf__max_depth'],
        class_weight='balanced', 
        random_state=42
    ))
])

best_rf_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val_rf = best_rf_model.predict(X_val)
print(classification_report(y_val, y_pred_val_rf))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test_rf = best_rf_model.predict(X_test)
print(classification_report(y_test, y_pred_test_rf))

--- Mengoptimasi Random Forest ---
Fitting 1 folds for each of 4 candidates, totalling 4 fits

Best RF Params: {'rf__max_depth': None, 'rf__n_estimators': 100, 'tfidf__max_features': 5000}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

      normal       0.95      0.98      0.96      9746
       toxic       0.70      0.46      0.55       996

    accuracy                           0.93     10742
   macro avg       0.82      0.72      0.76     10742
weighted avg       0.92      0.93      0.92     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

      normal       0.98      0.99      0.99      9262
       toxic       0.56      0.28      0.38       280

    accuracy                           0.97      9542
   macro avg       0.77      0.64      0.68      9542
weighted avg       0.97      0.97      0.97      9542



In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline

# Load dataset
df_train = pd.read_csv('../resources/train_preprocessing_awal_shallow.csv')
df_val = pd.read_csv('../resources/val_preprocessing_awal_shallow.csv')
df_test = pd.read_csv('../resources/test_preprocessing_awal_shallow.csv')

X_train = df_train['Pesan_Clean'].fillna('')
y_train = df_train['Label Judi Online']

X_val = df_val['Pesan_Clean'].fillna('')
y_val = df_val['Label Judi Online']

X_test = df_test['Pesan_Clean'].fillna('')
y_test = df_test['Label Judi Online']

In [8]:
# LinearSVM dengan GridSearchCV
X_combined = pd.concat([X_train, X_val], axis=0)
y_combined = pd.concat([y_train, y_val], axis=0)

split_index = [-1] * len(X_train) + [0] * len(X_val)
pds = PredefinedSplit(test_fold=split_index)

# Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('svm', LinearSVC(class_weight='balanced', random_state=42, max_iter=2000))
])

# Param
param_grid = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__max_features': [5000, 10000],
    'svm__C': [0.1, 1, 10]
}

print("Memulai GridSearchCV... (Ini akan mencoba banyak kombinasi)")
grid_search = GridSearchCV(pipeline, param_grid, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_search.fit(X_combined, y_combined) 

print(f"\nParameter Terbaik SVM: {grid_search.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_svm_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=grid_search.best_params_['tfidf__ngram_range'],
        max_features=grid_search.best_params_['tfidf__max_features']
    )),
    ('svm', LinearSVC(
        C=grid_search.best_params_['svm__C'],
        class_weight='balanced', 
        random_state=42, 
        max_iter=2000
    ))
])

best_svm_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val = best_svm_model.predict(X_val)
print(classification_report(y_val, y_pred_val))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test = best_svm_model.predict(X_test)
print(classification_report(y_test, y_pred_test))

Memulai GridSearchCV... (Ini akan mencoba banyak kombinasi)
Fitting 1 folds for each of 12 candidates, totalling 12 fits

Parameter Terbaik SVM: {'svm__C': 1, 'tfidf__max_features': 10000, 'tfidf__ngram_range': (1, 1)}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

 judi online       0.08      0.39      0.14       114
      normal       0.99      0.95      0.97     10628

    accuracy                           0.95     10742
   macro avg       0.54      0.67      0.56     10742
weighted avg       0.98      0.95      0.96     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

 judi online       0.14      0.51      0.22        95
      normal       0.99      0.97      0.98      9447

    accuracy                           0.96      9542
   macro avg       0.57      0.74      0.60      9542
weighted avg       0.99      0.96      0.9

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import classification_report
import pandas as pd

X_combined = pd.concat([X_train, X_val], axis=0)
y_combined = pd.concat([y_train, y_val], axis=0)

split_index = [-1] * len(X_train) + [0] * len(X_val)
pds = PredefinedSplit(test_fold=split_index)

In [10]:
print("--- Mengoptimasi Logistic Regression ---")
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('lr', LogisticRegression(class_weight='balanced', max_iter=1000))
])

param_lr = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'lr__C': [0.1, 1, 10],
    'lr__solver': ['liblinear', 'lbfgs']
}

grid_lr = GridSearchCV(pipe_lr, param_lr, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_lr.fit(X_combined, y_combined)

print(f"\nBest LR Params: {grid_lr.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_lr_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=grid_lr.best_params_['tfidf__ngram_range']
    )),
    ('lr', LogisticRegression(
        C=grid_lr.best_params_['lr__C'],
        solver=grid_lr.best_params_['lr__solver'],
        class_weight='balanced', 
        max_iter=1000
    ))
])

best_lr_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val_lr = best_lr_model.predict(X_val)
print(classification_report(y_val, y_pred_val_lr))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test_lr = best_lr_model.predict(X_test)
print(classification_report(y_test, y_pred_test_lr))

--- Mengoptimasi Logistic Regression ---
Fitting 1 folds for each of 12 candidates, totalling 12 fits

Best LR Params: {'lr__C': 10, 'lr__solver': 'liblinear', 'tfidf__ngram_range': (1, 1)}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

 judi online       0.09      0.32      0.14       114
      normal       0.99      0.96      0.98     10628

    accuracy                           0.96     10742
   macro avg       0.54      0.64      0.56     10742
weighted avg       0.98      0.96      0.97     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

 judi online       0.18      0.48      0.26        95
      normal       0.99      0.98      0.99      9447

    accuracy                           0.97      9542
   macro avg       0.59      0.73      0.63      9542
weighted avg       0.99      0.97      0.98      9542



In [11]:
print("--- Mengoptimasi Naive Bayes ---")
pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('nb', MultinomialNB())
])

param_nb = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'nb__alpha': [0.01, 0.1, 1, 10]
}

grid_nb = GridSearchCV(pipe_nb, param_nb, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_nb.fit(X_combined, y_combined)

print(f"\nBest NB Params: {grid_nb.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_nb_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=grid_nb.best_params_['tfidf__ngram_range']
    )),
    ('nb', MultinomialNB(
        alpha=grid_nb.best_params_['nb__alpha']
    ))
])

best_nb_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val_nb = best_nb_model.predict(X_val)
print(classification_report(y_val, y_pred_val_nb))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test_nb = best_nb_model.predict(X_test)
print(classification_report(y_test, y_pred_test_nb))

--- Mengoptimasi Naive Bayes ---
Fitting 1 folds for each of 8 candidates, totalling 8 fits

Best NB Params: {'nb__alpha': 0.1, 'tfidf__ngram_range': (1, 2)}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

 judi online       0.35      0.31      0.33       114
      normal       0.99      0.99      0.99     10628

    accuracy                           0.99     10742
   macro avg       0.67      0.65      0.66     10742
weighted avg       0.99      0.99      0.99     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

 judi online       0.40      0.42      0.41        95
      normal       0.99      0.99      0.99      9447

    accuracy                           0.99      9542
   macro avg       0.70      0.71      0.70      9542
weighted avg       0.99      0.99      0.99      9542



In [12]:
print("--- Mengoptimasi Random Forest ---")
pipe_rf = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('rf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_rf = {
    'tfidf__max_features': [5000], 
    'rf__n_estimators': [100, 200],
    'rf__max_depth': [None, 20]
}

grid_rf = GridSearchCV(pipe_rf, param_rf, cv=pds, scoring='f1_macro', verbose=1, refit=False)
grid_rf.fit(X_combined, y_combined)

print(f"\nBest RF Params: {grid_rf.best_params_}")

print("\n=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===")
best_rf_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=grid_rf.best_params_['tfidf__max_features']
    )),
    ('rf', RandomForestClassifier(
        n_estimators=grid_rf.best_params_['rf__n_estimators'],
        max_depth=grid_rf.best_params_['rf__max_depth'],
        class_weight='balanced', 
        random_state=42
    ))
])

best_rf_model.fit(X_train, y_train)

print("\n=== HASIL EVALUASI PADA DATA VALIDASI ===")
y_pred_val_rf = best_rf_model.predict(X_val)
print(classification_report(y_val, y_pred_val_rf))

print("\n=== HASIL EVALUASI PADA DATA TESTING ===")
y_pred_test_rf = best_rf_model.predict(X_test)
print(classification_report(y_test, y_pred_test_rf))

--- Mengoptimasi Random Forest ---
Fitting 1 folds for each of 4 candidates, totalling 4 fits

Best RF Params: {'rf__max_depth': None, 'rf__n_estimators': 200, 'tfidf__max_features': 5000}

=== MELATIH MODEL TERBAIK HANYA DENGAN DATA TRAIN ===

=== HASIL EVALUASI PADA DATA VALIDASI ===
              precision    recall  f1-score   support

 judi online       0.10      0.17      0.12       114
      normal       0.99      0.98      0.99     10628

    accuracy                           0.98     10742
   macro avg       0.55      0.58      0.56     10742
weighted avg       0.98      0.98      0.98     10742


=== HASIL EVALUASI PADA DATA TESTING ===
              precision    recall  f1-score   support

 judi online       0.18      0.20      0.19        95
      normal       0.99      0.99      0.99      9447

    accuracy                           0.98      9542
   macro avg       0.59      0.60      0.59      9542
weighted avg       0.98      0.98      0.98      9542

